# step 1 - 라이브러리 불러오기

In [1]:
import FinanceDataReader as fdr
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import unicodedata
import datetime

print('라이브러리 임포트 완료!')

라이브러리 임포트 완료!


# step 2 - KOSPI 전체 종목 리스트 조회

In [2]:
market = 'KOSPI'
df_market = fdr.StockListing(market)

print(f'전체 종목 수 : {len(df_market)}개')
print(f'컬럼 목록 : {df_market.columns.tolist()}') # 리스트형태로 보겠다

df_market.head()

전체 종목 수 : 948개
컬럼 목록 : ['Unnamed: 0', 'Code', 'ISU_CD', 'Name', 'Market', 'Dept', 'Close', 'ChangeCode', 'Changes', 'ChagesRatio', 'Open', 'High', 'Low', 'Volume', 'Amount', 'Marcap', 'Stocks', 'MarketId']


,Unnamed: 0,Code,ISU_CD,Name,Market,Dept,Close,ChangeCode,Changes,ChagesRatio,Open,High,Low,Volume,Amount,Marcap,Stocks,MarketId
0,0,005930,KR7005930003,삼성전자,KOSPI,NaN,284000,1,5000,1.79,264000,285500,262000,35540134,9797838529400,1660343124672000,5846278608,STK
1,1,000660,KR7000660001,SK하이닉스,KOSPI,NaN,1976000,1,141000,7.68,1781000,1990000,1781000,7126921,13535538165412,1408299873240000,712702365,STK
2,2,402340,KR7402340004,SK스퀘어,KOSPI,NaN,1190000,1,64000,5.68,1071000,1193000,1071000,1015340,1169118518000,157030479340000,131958386,STK
3,3,005935,KR7005931001,삼성전자우,KOSPI,NaN,189100,1,2100,1.12,174200,190500,174100,6046834,1102809839600,151728394487300,802371203,STK
4,4,005380,KR7005380001,현대차,KOSPI,NaN,710000,1,64000,9.91,642000,710000,642000,4051665,2785128978700,145378013860000,204757766,STK


# step 3 - 한글 종목명 정규화

In [3]:
# 정규화 전후 비교 예시
# s1 = '삼성전자'           # NFC (일반적인 한글 입력)
# s2 = '\u삼\u성\u전\u자'  # NFD (분해된 형태, 눈에는 같아 보임)

def normalize_str(s):
    return unicodedata.normalize('NFKC', s).strip()

# 전각 문자 예시 (실제로 종목명에서 발생할 수 있는 케이스)
test_cases = [
    ('㈜카카오', '정규화 전'),
    (normalize_str('㈜카카오'), '정규화 후'),
]

for text, label in test_cases:
    print(f'{label}: {text}')

# 실제 적용: df_market 종목명 전체 정규화
df_market['Name'] = df_market['Name'].apply(normalize_str)
print('\n✅ 종목명 정규화 완료')
df_market[['Code', 'Name', 'Marcap']].head()

정규화 전: ㈜카카오
정규화 후: (주)카카오

✅ 종목명 정규화 완료


,Code,Name,Marcap
0,005930,삼성전자,1660343124672000
1,000660,SK하이닉스,1408299873240000
2,402340,SK스퀘어,157030479340000
3,005935,삼성전자우,151728394487300
4,005380,현대차,145378013860000


# step 4 - 시가총액 TOP 10 추출

In [4]:
top10 = df_market.nlargest(10, 'Marcap').iloc[::-1]  # 반전

# 시가총액 단위 변환 : 원 -> 조 (1조 --> 10^12)
top10_display = top10[['Name', 'Marcap']].copy()
top10_display['시가총액(조)'] = (top10_display['Marcap'] / 1e12).round(1)

top10_display[['Name', '시가총액(조)']].reset_index(drop=True)

,Name,시가총액(조)
0,기아,70.1
1,삼성전기,76.9
2,두산에너빌리티,76.9
3,HD현대중공업,76.9
4,LG에너지솔루션,100.6
5,현대차,145.4
6,삼성전자우,151.7
7,SK스퀘어,157.0
8,SK하이닉스,1408.3
9,삼성전자,1660.3


# step 5 - 시가총액 TOP 10 수평막대그래프 

In [5]:
fig_top10 = go.Figure(go.Bar(
    x=top10['Marcap'] / 1e12,
    y=top10['Name'],
    orientation='h',  # 수평 막대그래프(가로)
    text=top10['Marcap'] / 1e12,
    texttemplate='%{text:.1f}조',  # 소수 첫째자리까지, + '조' 단위
    marker_color='steelblue'  # 막대 색상
))

fig_top10.update_layout(
    title=f'{market} 시가총액 TOP 10',
    xaxis_title='시가총액 (조)',
    yaxis_title='종목명',
    bargap=0.15,   # 막대 사이 간격 (0~1)
    height=450
)

fig_top10.show()

# step 6 - 개별 종목 주가 데이터 조회

In [6]:
# 종목명 -> 종목코드 변환
target_name = '삼성전자'
code = df_market.loc[df_market['Name'] == target_name, 'Code'].values[0]
print(f'{target_name} 종목코드 : {code}')

삼성전자 종목코드 : 005930


In [7]:
# 주가 데이터 조회
start_date = '2024-01-01'
end_date = datetime.datetime.now().strftime('%Y-%m-%d')

df_stock = fdr.DataReader(code, start_date, end_date)

print(f'조회 기간 : {start_date} ~ {end_date}')
print(f'데이터 수 : {len(df_stock)}일')
df_stock.tail() # 최근 5일

조회 기간 : 2024-01-01 ~ 2026-05-13
데이터 수 : 574일


,Open,High,Low,Close,Volume,Change
Date,,,,,,
2026-05-07,272000,277000,260000,271500,41404687,0.020677
2026-05-08,260000,270000,260000,268500,25875880,-0.011050
2026-05-11,284500,288500,280000,285500,36031094,0.063315
2026-05-12,290000,291500,266000,279000,41211149,-0.022767
2026-05-13,264000,285500,262000,284000,35435143,0.017921


# step 7 - 현재가 / 전일 대비 확인

In [8]:
current = df_stock['Close'].iloc[-1]  # 최근 종가
prev = df_stock['Close'].iloc[-2]  # 전일 종가
delta = current - prev   # 전일 대비 변동폭
pct = delta / prev * 100   # 등락률 (%)

direction = '▲' if delta > 0 else '▼'

print(f'[ {target_name} ({code}) ]')
print(f'현재가 : {current:,}원')
print(f'전일 대비 : {direction} {abs(delta):,}원  ({pct:+.2f}%)')

[ 삼성전자 (005930) ]
현재가 : 284,000원
전일 대비 : ▲ 5,000원  (+1.79%)


# step 8 - 종가 라인 차트 (단일 종목)

In [9]:
fig_line = px.line(
    df_stock,
    y='Close',
    title=f'{target_name} 종가 추이',
    labels={'Close':'종가 (원)', 'Date':'날짜'}
)

fig_line.update_layout(height=400)
fig_line.show()

# step 9 - 다중 종목 종가 비교

In [10]:
# 비교할 종목 목록
compare_names = ['삼성전자', 'SK하이닉스', 'LG에너지솔루션']

dfs = []
for name in compare_names:
    matched = df_market.loc[df_market['Name'] == name, 'Code'].values
    if len(matched) == 0:
        print(f'{name}: 종목 코드를 찾을 수 없습니다.')
        continue

    c = matched[0]
    df_temp = fdr.DataReader(c, start_date, end_date)

    if not df_temp.empty:
        # Close 열만 추출하고, 열 이름을 종목명으로 변경
        dfs.append(df_temp[['Close']].rename(columns={'Close':name}))
        print(f'{name}({c}) 데이터 로드 완료 ({len(df_temp)}일)')

# 수평 병합
merged_df = pd.concat(dfs, axis=1)        
print(f'병합 결과 : {merged_df.shape}')
merged_df.tail()

삼성전자(005930) 데이터 로드 완료 (574일)
SK하이닉스(000660) 데이터 로드 완료 (574일)
LG에너지솔루션(373220) 데이터 로드 완료 (574일)
병합 결과 : (574, 3)


,삼성전자,SK하이닉스,LG에너지솔루션
Date,,,
2026-05-07,271500,1654000,483000
2026-05-08,268500,1686000,476500
2026-05-11,285500,1880000,468000
2026-05-12,279000,1835000,443000
2026-05-13,284000,1976000,430000


In [11]:

# 다중 종목 라인 차트
fig_multi = px.line(
    merged_df,
    title='주요 종목 종가 비교',
    labels={'value': '종가 (원)', 'Date': '날짜', 'variable': '종목'}
)

fig_multi.update_layout(height=450, legend_title='종목')
fig_multi.show()

# step 10 캔들스틱 차트
- go.Candlestick(
    x=날짜,
    open=시가,
    high=고가,
    low=저가,
    close=종가
)

## df_stock.index[-1]  --> 가장 최근 날짜

## pd.DateOffset(months=3) --> 달력 기준 3개월
- datetime.timedelta(days=90)과 비슷하다.
- 월말 처리 부분이 더 정확하다!

In [12]:
# 최근 3개월만 표시 
three_months_ago = df_stock.index[-1] - pd.DateOffset(months=3)
# 결과 데이터프레임
df_candle = df_stock[df_stock.index >= three_months_ago]

In [ ]:
# 결과 차트 정보 저장하기 전에 데이터확인
df_candle.tail()

,Open,High,Low,Close,Volume,Change
Date,,,,,,
2026-05-07,272000,277000,260000,271500,41404687,0.020677
2026-05-08,260000,270000,260000,268500,25875880,-0.011050
2026-05-11,284500,288500,280000,285500,36031094,0.063315
2026-05-12,290000,291500,266000,279000,41211149,-0.022767
2026-05-13,264000,285500,262000,284000,35435143,0.017921


In [14]:
df_candle.index

DatetimeIndex(['2026-02-13', '2026-02-19', '2026-02-20', '2026-02-23',
               '2026-02-24', '2026-02-25', '2026-02-26', '2026-02-27',
               '2026-03-03', '2026-03-04', '2026-03-05', '2026-03-06',
               '2026-03-09', '2026-03-10', '2026-03-11', '2026-03-12',
               '2026-03-13', '2026-03-16', '2026-03-17', '2026-03-18',
               '2026-03-19', '2026-03-20', '2026-03-23', '2026-03-24',
               '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-30',
               '2026-03-31', '2026-04-01', '2026-04-02', '2026-04-03',
               '2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09',
               '2026-04-10', '2026-04-13', '2026-04-14', '2026-04-15',
               '2026-04-16', '2026-04-17', '2026-04-20', '2026-04-21',
               '2026-04-22', '2026-04-23', '2026-04-24', '2026-04-27',
               '2026-04-28', '2026-04-29', '2026-04-30', '2026-05-04',
               '2026-05-06', '2026-05-07', '2026-05-08', '2026-05-11',
      

In [15]:
# 캔들 차트
fig_candle = go.Figure(data=[go.Candlestick(
    x=df_candle.index,  # 날짜
    open=df_candle['Open'],
    high=df_candle['High'],
    low=df_candle['Low'],
    close=df_candle['Close']
)])

# 캔들 차트 - 디자인
fig_candle.update_layout(
    title=f'{target_name} 캔들스틱 차트(최근3개월)',
    xaxis_title='날짜',
    yaxis_title='가격 (원)',
    xaxis_rangeslider_visible=False, # 하단 범위 슬라이더 숨김,
    height=500
)

fig_candle.show()